# Notebook 6 — Grad-CAM: Visual Explainability for RA Classification

**Project:** P63 – Multimodal Deep Learning for Autoimmune Disease Diagnosis  
**Phase:** Rheumatoid Arthritis (RA) — Explainability / XAI  

---

## What is Grad-CAM?

**Gradient-weighted Class Activation Mapping (Grad-CAM)** is an XAI (Explainable AI) technique that produces a heatmap showing *which regions of an input image* the neural network focused on when making its prediction.

For a clinical application like RA diagnosis, this is critically important:
- It helps verify that the model is looking at *clinically relevant* regions (e.g. joint spaces, bone erosions) rather than image artefacts or scanner characteristics.
- It provides interpretability for medical professionals who need to understand and trust the model's decisions.
- It is a standard technique in medical AI research and expected in an academic FYP.

### How it works (simplified)
1. Run a forward pass through the network to get a prediction.
2. Compute the gradient of the predicted class score with respect to the activations of the **last convolutional layer** (`backbone.layer4` in ResNet-18).
3. Average the gradients spatially to get per-channel importance weights.
4. Compute a weighted sum of the activation maps.
5. Apply ReLU (keep only positive activations).
6. Upsample to the original image size and overlay as a heatmap.

### Reference
Selvaraju et al. (2017). "Grad-CAM: Visual Explanations from Deep Networks via Gradient-based Localization."  
ICCV 2017. https://arxiv.org/abs/1610.02391

---

## Status of this notebook

**This notebook is prepared for future implementation.**

The full implementation requires a trained model checkpoint (`models/ra_resnet18_best.pth`).  
Run `notebooks/05_train_ra_cnn.ipynb` first, then return here.

The Grad-CAM class is fully implemented below — it will produce real heatmaps once the trained model is available. **No fake or placeholder heatmaps are generated.**

---
## Section 1 — Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT:', PROJECT_ROOT)

---
## Section 2 — Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm

import torch
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T

from src.utils   import get_device, load_norm_stats, BEST_MODEL, PLOTS_DIR, TEST_CSV, CLASS_NAMES
from src.model   import build_model
from src.dataset import get_transforms

print('Imports OK.')

---
## Section 3 — Configuration

In [ ]:
DEVICE       = get_device()
IMG_SIZE     = (224, 224)
N_EXAMPLES   = 6    # number of test images to visualise
GRADCAM_LAYER = 'backbone.layer4'   # last conv block of ResNet-18

print(f'Device          : {DEVICE}')
print(f'Target layer    : {GRADCAM_LAYER}')
print(f'Examples to show: {N_EXAMPLES}')

---
## Section 4 — Grad-CAM Implementation

The `GradCAM` class uses **PyTorch hooks** — callback functions registered on a specific layer that capture the layer's output (activations) and the gradient flowing back through it.

This approach requires **no changes to the model architecture** — it is non-invasive and works with any ResNet variant.

In [ ]:
class GradCAM:
    """
    Gradient-weighted Class Activation Mapping for ResNet-based models.

    Works by registering forward and backward hooks on the target layer
    to capture activations and gradients during inference.

    Parameters
    ----------
    model      : trained RAClassifier
    target_layer: the nn.Module layer to hook (use model.backbone.layer4)

    Usage
    -----
    gradcam = GradCAM(model, model.backbone.layer4)
    heatmap = gradcam.generate(image_tensor, class_idx=1)  # 1 = RA
    gradcam.remove_hooks()  # always clean up after use
    """

    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer

        # Storage for the captured values
        self._activations = None
        self._gradients   = None

        # Register hooks
        # A forward hook captures the layer's OUTPUT during the forward pass.
        self._fwd_hook = target_layer.register_forward_hook(self._save_activation)
        # A backward hook captures the GRADIENT flowing back through the layer.
        self._bwd_hook = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        """Called automatically during the forward pass."""
        self._activations = output.detach()   # shape: (1, C, H', W')

    def _save_gradient(self, module, grad_input, grad_output):
        """Called automatically during the backward pass."""
        self._gradients = grad_output[0].detach()  # shape: (1, C, H', W')

    def generate(
        self,
        image_tensor: torch.Tensor,
        class_idx:    int = None,
    ) -> np.ndarray:
        """
        Generate a Grad-CAM heatmap for one image.

        Parameters
        ----------
        image_tensor : (1, 3, 224, 224) float32 tensor on the correct device
        class_idx    : which class to explain (None = use the predicted class)

        Returns
        -------
        heatmap : np.ndarray of shape (224, 224), values in [0, 1]
                  Brighter regions = more important for the prediction.
        """
        self.model.eval()

        # ── Forward pass ──────────────────────────────────────────────────────
        image_tensor = image_tensor.to(next(self.model.parameters()).device)
        image_tensor.requires_grad_(False)

        # We need gradients with respect to the activations, not the input
        logits = self.model(image_tensor)   # shape: (1, 2)

        # If no class specified, use the predicted class
        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()

        # ── Backward pass ──────────────────────────────────────────────────────
        # Zero all existing gradients
        self.model.zero_grad()

        # Compute gradient of the class score w.r.t. the target layer activations
        # We use a one-hot vector to select only the gradient for class_idx
        one_hot = torch.zeros_like(logits)
        one_hot[0, class_idx] = 1.0
        logits.backward(gradient=one_hot, retain_graph=False)

        # ── Compute Grad-CAM ───────────────────────────────────────────────────
        # activations shape: (1, C, H', W') — feature maps from last conv layer
        # gradients shape:   (1, C, H', W') — gradient of class score w.r.t. each map
        activations = self._activations.squeeze(0)   # (C, H', W')
        gradients   = self._gradients.squeeze(0)     # (C, H', W')

        # Global average pool the gradients → importance weight per channel
        weights = gradients.mean(dim=(1, 2))         # shape: (C,)

        # Weighted sum of activation maps
        cam = torch.zeros(activations.shape[1:], dtype=torch.float32)  # (H', W')
        for i, w in enumerate(weights):
            cam += w * activations[i]   # add weighted activation map

        # ReLU: keep only positive activations (regions that support the class)
        cam = F.relu(cam)

        # Upsample to the original image size (224, 224)
        cam = cam.unsqueeze(0).unsqueeze(0)   # (1, 1, H', W')
        cam = F.interpolate(
            cam,
            size  = (224, 224),
            mode  = 'bilinear',
            align_corners = False,
        )
        cam = cam.squeeze().numpy()   # (224, 224)

        # Normalise to [0, 1]
        cam_min, cam_max = cam.min(), cam.max()
        if cam_max - cam_min > 1e-8:
            cam = (cam - cam_min) / (cam_max - cam_min)
        else:
            cam = np.zeros_like(cam)  # flat activation → no meaningful heatmap

        return cam, class_idx

    def remove_hooks(self):
        """Remove the registered hooks. Always call this after you are done."""
        self._fwd_hook.remove()
        self._bwd_hook.remove()


print('GradCAM class defined.')

---
## Section 5 — Overlay Utility

In [ ]:
def overlay_heatmap(
    image_rgb: np.ndarray,
    heatmap:   np.ndarray,
    alpha:     float = 0.45,
    colormap:  str   = 'jet',
) -> np.ndarray:
    """
    Overlay a Grad-CAM heatmap on the original image.

    Parameters
    ----------
    image_rgb : (H, W, 3) numpy array, uint8 in [0, 255]
    heatmap   : (H, W) numpy array, float in [0, 1]
    alpha     : blending factor (0=image only, 1=heatmap only)
    colormap  : matplotlib colormap name (default 'jet')

    Returns
    -------
    blended : (H, W, 3) numpy array, uint8
    """
    # Apply the colormap to the heatmap → RGB colours
    cmap   = cm.get_cmap(colormap)
    hmap_c = (cmap(heatmap)[:, :, :3] * 255).astype(np.uint8)  # (H, W, 3)

    # Blend: weighted sum of image and coloured heatmap
    blended = ((1 - alpha) * image_rgb + alpha * hmap_c).astype(np.uint8)
    return blended


print('overlay_heatmap() defined.')

---
## Section 6 — Load Trained Model

**Prerequisite:** Run `notebooks/05_train_ra_cnn.ipynb` to generate `models/ra_resnet18_best.pth`.

In [ ]:
if not BEST_MODEL.exists():
    print('=' * 60)
    print('MODEL CHECKPOINT NOT FOUND')
    print(f'Expected: {BEST_MODEL}')
    print()
    print('This notebook requires a trained model.')
    print('Please run notebooks/05_train_ra_cnn.ipynb first.')
    print('Then re-run this notebook.')
    print('=' * 60)
    MODEL_READY = False
else:
    model = build_model(
        mode       = 'image_only',
        pretrained = False,   # weights loaded from checkpoint, not ImageNet
        device     = DEVICE,
    )
    checkpoint = torch.load(BEST_MODEL, map_location=DEVICE, weights_only=True)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()
    print(f'Loaded best model from: {BEST_MODEL}')
    print(f'  Checkpoint epoch : {checkpoint["epoch"]}')
    print(f'  Val loss         : {checkpoint["val_loss"]:.4f}')
    MODEL_READY = True

---
## Section 7 — Generate Grad-CAM Heatmaps

In [ ]:
if not MODEL_READY:
    print('Skipping Grad-CAM — trained model not available.')
    print('Run Notebook 05 to train the model, then re-run this cell.')
else:
    # Load test manifest
    test_df = pd.read_csv(TEST_CSV)
    mean, std = load_norm_stats()
    transform = get_transforms('test', IMG_SIZE, mean, std)

    # Select examples: 3 RA + 3 Non-RA
    ra_samples    = test_df[test_df['isRA'] == 1].sample(min(3, (test_df['isRA']==1).sum()), random_state=42)
    nonra_samples = test_df[test_df['isRA'] == 0].sample(min(3, (test_df['isRA']==0).sum()), random_state=42)
    examples = pd.concat([nonra_samples, ra_samples], ignore_index=True)

    print(f'Generating Grad-CAM for {len(examples)} test images...')
    print(f'Target layer: {GRADCAM_LAYER}')
    print()

    # Register Grad-CAM hooks on the last residual block of ResNet-18
    gradcam = GradCAM(model, model.backbone.layer4)

    results = []   # store (original_rgb, heatmap, pred_class, true_class, filename)

    for _, row in examples.iterrows():
        # ── Load original image (for display) ────────────────────────────────
        with Image.open(row['full_path']) as img:
            img_rgb = img.convert('RGB').resize(IMG_SIZE)
            img_arr = np.array(img_rgb)   # (224, 224, 3) uint8

        # ── Preprocess for model input ────────────────────────────────────────
        img_tensor = transform(img.convert('RGB')).unsqueeze(0)  # (1, 3, 224, 224)

        # ── Generate Grad-CAM heatmap ─────────────────────────────────────────
        # We explain the predicted class (not forced to the true class).
        # This shows what the model is actually looking at for ITS decision.
        heatmap, pred_idx = gradcam.generate(img_tensor, class_idx=None)

        pred_label = CLASS_NAMES[pred_idx]
        true_label = CLASS_NAMES[int(row['isRA'])]
        correct    = '✓' if pred_idx == int(row['isRA']) else '✗'

        results.append({
            'img_arr'    : img_arr,
            'heatmap'    : heatmap,
            'pred_label' : pred_label,
            'true_label' : true_label,
            'correct'    : correct,
            'filename'   : row['filename'],
        })
        print(f'  {row["filename"][:40]:40s}  true={true_label:6s}  pred={pred_label:6s}  {correct}')

    gradcam.remove_hooks()  # always clean up hooks after use
    print()
    print('Grad-CAM generation complete.')

---
## Section 8 — Visualise Heatmaps

In [ ]:
if not MODEL_READY:
    print('Skipping visualisation — trained model not available.')
else:
    n = len(results)
    fig, axes = plt.subplots(n, 3, figsize=(15, 4.5 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(
        'Grad-CAM Heatmaps — RA Classification\n'
        'Column 1: Original X-ray  |  Column 2: Heatmap  |  Column 3: Overlay',
        fontsize=13, fontweight='bold',
    )

    for i, res in enumerate(results):
        img_arr   = res['img_arr']
        heatmap   = res['heatmap']
        overlay   = overlay_heatmap(img_arr, heatmap, alpha=0.45)
        title_str = (f"{res['filename'][:35]}\n"
                     f"True: {res['true_label']}  |  Pred: {res['pred_label']}  {res['correct']}")

        # Original image
        axes[i][0].imshow(img_arr)
        axes[i][0].set_title(title_str, fontsize=8)
        axes[i][0].axis('off')

        # Heatmap only
        axes[i][1].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
        axes[i][1].set_title('Grad-CAM heatmap', fontsize=9)
        axes[i][1].axis('off')

        # Overlay
        axes[i][2].imshow(overlay)
        axes[i][2].set_title('Overlay (image + heatmap)', fontsize=9)
        axes[i][2].axis('off')

    plt.tight_layout()
    save_path = PLOTS_DIR / 'gradcam_examples.png'
    plt.savefig(save_path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Grad-CAM visualisation saved → {save_path}')

---
## Section 9 — Save Individual Heatmaps

In [ ]:
if not MODEL_READY:
    print('Skipping — trained model not available.')
else:
    gradcam_dir = PLOTS_DIR / 'gradcam'
    gradcam_dir.mkdir(exist_ok=True)

    for i, res in enumerate(results):
        overlay = overlay_heatmap(res['img_arr'], res['heatmap'], alpha=0.45)
        stem    = Path(res['filename']).stem
        fname   = f'gradcam_{stem}_{res["true_label"]}.png'
        Image.fromarray(overlay).save(gradcam_dir / fname)

    print(f'Individual Grad-CAM overlays saved → {gradcam_dir}')
    print(f'Files: {len(results)}')

---
## Section 10 — Interpretation Guide

### How to interpret a Grad-CAM heatmap for RA

| Colour | Meaning |
|---|---|
| Red / warm  | High importance — the model focused here |
| Blue / cool | Low importance — the model mostly ignored this region |

**What to look for in RA radiographs:**
- The model should focus on **joint spaces** (the narrow gaps between bones).
- It should attend to **periarticular bones** at the wrist and finger joints.
- In RA, radiographic hallmarks include:
  - Periarticular osteopenia (reduced bone density near joints)
  - Joint space narrowing
  - Marginal bone erosions (especially at MCP and PIP joints)

**Red flags (model may not be reliable if):**
- The heatmap focuses on image borders, background, or non-anatomical regions.
- The heatmap looks uniform (no clear focus area).
- The model makes correct predictions but focuses on uninformative areas — this can indicate the model learned spurious correlations (e.g. imaging site characteristics).

### Next steps for Grad-CAM analysis
1. Run this notebook after full training (not smoke-test weights).
2. Inspect heatmaps for both correct and incorrect predictions.
3. Show heatmaps to a clinician to validate that the model focuses on relevant anatomy.
4. Consider `GuidedBackpropagation` or `ScoreCAM` as complementary methods.

---
## Section 11 — Batch Heatmap Generation (Future Extension)

The cell below is a ready-to-use template for generating heatmaps for the **entire test set** and saving them systematically. This is useful for a quantitative XAI analysis (e.g. measuring how much of the heatmap falls on the joint region).

In [ ]:
# ── FUTURE EXTENSION: batch Grad-CAM over entire test set ─────────────────────
# Uncomment and run after full training is complete.

# if MODEL_READY:
#     from torch.utils.data import DataLoader
#     from src.dataset import RADataset
#
#     test_dataset = RADataset(TEST_CSV, transform=get_transforms('test', IMG_SIZE, mean, std))
#     batch_dir    = PLOTS_DIR / 'gradcam_batch'
#     batch_dir.mkdir(exist_ok=True)
#
#     gc_batch = GradCAM(model, model.backbone.layer4)
#
#     for idx in range(len(test_dataset)):
#         img_t, meta_t, label = test_dataset[idx]
#         row = test_dataset.df.iloc[idx]
#
#         hmap, pred = gc_batch.generate(img_t.unsqueeze(0))
#
#         with Image.open(row['full_path']) as img:
#             img_arr = np.array(img.convert('RGB').resize(IMG_SIZE))
#         overlay = overlay_heatmap(img_arr, hmap)
#
#         stem  = Path(row['filename']).stem
#         fname = f'{stem}_true{int(label)}_pred{pred}.png'
#         Image.fromarray(overlay).save(batch_dir / fname)
#
#     gc_batch.remove_hooks()
#     print(f'Batch Grad-CAM saved to {batch_dir}')

print('Batch Grad-CAM template is ready (currently commented out).')
print('Uncomment after full training is complete.')